# Phase 6 - Step 3: decoding against repetition (no retraining)

P6_02 result: every model **loops** with greedy decoding ("The respondent company was a limited company ... was a limited
one ... was a limited owner ..."): 22-32% of the 4-grams in our summaries are repeats (references: ~3%), and the first
repeated 6-word sequence appears at word ~45-73. Blocking only repeated 8-token sequences was too weak: the model dodges
with tiny variations.

Decoding is applied AFTER training, so we can fix it without retraining (doc 12.1: "decoding strategies",
"redundancy reduction"). Settings compared (all greedy):

| tag         | no repeated n-token sequence in the summary            | repetition penalty on tokens already in the summary |
| ----------- | ------------------------------------------------------ | --------------------------------------------------- |
| `nr8`       | 8 (P6_02 setting)                                      | -                                                   |
| `nr4`       | 4                                                      | -                                                   |
| `nr3`       | 3 (the usual choice for news summarisation, e.g. BART) | -                                                   |
| `nr8_rp1.3` | 8                                                      | 1.3                                                 |
| `nr4_rp1.2` | 4                                                      | 1.2                                                 |

The repetition penalty (Keskar et al. 2019) makes a token that is already in the summary less likely; tokens of the
document that are copied for the first time are NOT penalised (unlike Phase 1, where the penalty also covered the prompt).

1. **Sweep** on the first 50 local-eval documents for the two best models (#1 A2 full FT, #5 D + fusion-in-decoder),
   each setting also scored when the summary is cut to 256 / 320 words (P6_02: #1 had its best BERTScore at ~256 words).
2. **Choose** the setting (+ cut) with the best mean BERTScore F1 of the two models.
3. **Apply** it to all experiments: full local-eval (150) + test (100) -> new `submission.zip` in `checkpoints/phase6/<run>/test_<tag>/`.

Cached like everything else (summaries saved every 10 documents). ~30 min sweep + ~2 h final generation.


In [1]:
import os, sys, json, time, traceback, itertools
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import torch
import summ_utils as su
from summ_experiment import Experiment
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
QUICK_TEST = os.environ.get("P6_QUICK_TEST") == "1"
QUICK = dict(train_limit=48, val_limit=8, eval_limit=4, test_limit=3, epochs=1, ckpt_every=2, max_words=40) if QUICK_TEST else {}
N_SWEEP = 3 if QUICK_TEST else 50
print("project root:", ROOT, "| cuda", torch.cuda.is_available(), "| QUICK_TEST", QUICK_TEST)

TEACHER = {"phase1": "A2", "method": "full", **QUICK}
EXPERIMENTS = [   # same configs as P6_02 (the trained models are loaded, nothing is retrained)
    ("#1 A2 full FT", TEACHER, True),
    ("#2 B + LoRMA", {"phase1": "B", "method": "lorma", **QUICK}, True),
    ("#3 A + Houlsby", {"phase1": "A", "method": "houlsby", **QUICK}, True),
    ("#4 D + LoRA 1x1024", {"phase1": "D", "method": "lora", **QUICK}, True),
    ("#5 D + LoRA 3x1024 (FiD)", {"phase1": "D", "method": "lora", "n_chunks": 3, **QUICK}, True),
    ("#6 D-small + KD", {"phase1": "D", "method": "full", "keep_dec_layers": [0, 2, 4], "epochs": 4, "kd_teacher": TEACHER, **QUICK}, True),
    ("#6c D-small, no KD", {"phase1": "D", "method": "full", "keep_dec_layers": [0, 2, 4], "epochs": 4, **QUICK}, False),
]
if QUICK_TEST:   # QUICK sets epochs=1 for every run (as in the P6_02 quick test)
    EXPERIMENTS = [(l, {**c, "epochs": 1}, s) for l, c, s in EXPERIMENTS]
    TEACHER["epochs"] = 1
SETTINGS = {
    "nr8": {"no_repeat": 8},
    "nr4": {"no_repeat": 4},
    "nr3": {"no_repeat": 3},
    "nr8_rp1.3": {"no_repeat": 8, "rep_penalty": 1.3},
    "nr4_rp1.2": {"no_repeat": 4, "rep_penalty": 1.2},
}
CUTS = [None, 256, 320]
SWEEP_MODELS = [EXPERIMENTS[0], EXPERIMENTS[4]]

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project | cuda True | QUICK_TEST False


## 1. Sweep (first 50 local-eval documents, models #1 and #5)

`nr8` = the P6_02 summaries (copied, not regenerated). For each setting the summaries are generated once; the cut
versions (256 / 320 words) are scored from the same summaries.


In [2]:
SWEEP_FILE = ROOT / "experiments" / "phase6" / "decoding_sweep.json"
sweep = json.load(open(SWEEP_FILE)) if SWEEP_FILE.exists() else {}
for label, cfg, _ in SWEEP_MODELS:
    exp = Experiment(ROOT, cfg)
    ids = exp.local_ids[:N_SWEEP]
    refs = [su.clean_summ(exp.by_id[i]["summary"]) for i in ids]
    # nr8 = P6_02 summaries: copy them so they are not generated again
    old, new = exp.dir / "local_preds.json", exp.dir / "local_preds_nr8.json"
    if old.exists() and not new.exists():
        p = json.load(open(old))
        json.dump({i: p[i] for i in ids if i in p}, open(new, "w"), indent=1)
    for tag, dec in SETTINGS.items():
        preds = exp._generate([(i, exp.by_id[i]["document"], exp.enc[i][0]) for i in ids],
                              exp.dir / f"local_preds_{tag}.json", f"{label} {tag}", dec)
        for cut in CUTS:
            key = f"{exp.name}|{tag}|{cut}"
            if key in sweep:
                continue
            hyp = [Experiment.cut(preds[i]["summary"], cut) for i in ids]
            sc = su.score_summaries(hyp, refs)
            sc["repeated_4gram_pct"] = 100 * float(np.mean([su.repetition_rate(h, 4) for h in hyp]))
            sweep[key] = sc
            json.dump(sweep, open(SWEEP_FILE, "w"), indent=1)
    del exp
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

rows = [{"model": k.split("|")[0], "setting": k.split("|")[1], "cut": k.split("|")[2], **v} for k, v in sweep.items()]
sw = pd.DataFrame(rows)
display(sw.pivot_table(index=["setting", "cut"], columns="model", values="bertscore_f1").round(4).rename_axis(columns="BERTScore F1"))
display(sw.pivot_table(index=["setting", "cut"], columns="model", values=["rouge1_f1", "repeated_4gram_pct", "pred_words"]).round(3))

data: train 6661, val 367, local_eval 150, test 100 | targets cut at 620 tokens: 62%
experiment: A2_full_doc900 | device cuda | precision torch.bfloat16


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 263.23it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 401.95it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

[cache] training already done -> loaded best epoch 2


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 209.88it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 263.79it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

experiment: D_lorar8_enc3x1024 | device cuda | precision torch.bfloat16


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 426.29it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 432.69it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

[cache] training already done -> loaded best epoch 4


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 218.68it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 351.80it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

BERTScore F1    A2_full_doc900  D_lorar8_enc3x1024
setting   cut                                     
nr3       256           0.8458              0.8494
          320           0.8466              0.8509
          None          0.8451              0.8504
nr4       256           0.8449              0.8487
          320           0.8456              0.8505
          None          0.8442              0.8495
nr4_rp1.2 256           0.8458              0.8526
          320           0.8464              0.8533
          None          0.8460              0.8521
nr8       256           0.8394              0.8426
          320           0.8386              0.8429
          None          0.8371              0.8415
nr8_rp1.3 256           0.8421              0.8431
          320           0.8417              0.8435
          None          0.8399              0.8409

pred_words                    repeated_4gram_pct                         rouge1_f1                   
model          A2_full_doc900 D_lorar8_enc3x1024     A2_full_doc900 D_lorar8_enc3x1024 A2_full_doc900 D_lorar8_enc3x1024
setting   cut                                                                                                           
nr3       256          242.32             239.44              0.000              0.000          0.339              0.354
          320          286.78             300.28              0.000              0.000          0.365              0.400
          None         341.08             433.26              0.000              0.000          0.379              0.453
nr4       256          241.04             241.88              0.000              0.000          0.330              0.352
          320          290.16             299.22              0.000              0.000          0.358              0.389
          None         352.46             419.48              0.000              0.013          0.381              0.441
nr4_rp1.2 256          242.20             241.02              0.000              0.000          0.332              0.363
          320          283.12             294.66              0.000              0.000          0.357              0.400
          None         361.18             405.34              0.000              0.000          0.387              0.449
nr8       256          242.20             237.04             16.970             17.415          0.320              0.326
          320          284.06             300.18             19.355             19.290          0.337              0.361
          None         350.86             430.46             21.847             22.011          0.358              0.406
nr8_rp1.3 256          241.08             240.30             13.543             13.886          0.325              0.332
          320          293.60             296.12             15.983             15.451          0.348              0.364
          None         366.28             421.52             18.803             17.689          0.376              0.402

## 2. Choose the setting: best mean BERTScore F1 over the two models


In [3]:
mean_bs = sw.groupby(["setting", "cut"])["bertscore_f1"].mean().sort_values(ascending=False)
display(mean_bs.round(4).to_frame("mean BERTScore F1 (#1, #5)"))
best_setting, best_cut = mean_bs.index[0]
BEST_DEC = {**SETTINGS[best_setting], "cut_words": None if best_cut == "None" else int(best_cut)}
BEST_TAG = f"{best_setting}_cut{best_cut}"
print("chosen decoding:", BEST_TAG, BEST_DEC)
json.dump({"tag": BEST_TAG, "decode": BEST_DEC}, open(ROOT / "experiments" / "phase6" / "decoding_choice.json", "w"), indent=1)

mean BERTScore F1 (#1, #5)
setting   cut                             
nr4_rp1.2 320                       0.8498
          256                       0.8492
          None                      0.8490
nr3       320                       0.8488
nr4       320                       0.8480
nr3       None                      0.8478
          256                       0.8476
nr4       None                      0.8469
          256                       0.8468
nr8_rp1.3 256                       0.8426
          320                       0.8426
nr8       256                       0.8410
          320                       0.8408
nr8_rp1.3 None                      0.8404
nr8       None                      0.8393

chosen decoding: nr4_rp1.2_cut320 {'no_repeat': 4, 'rep_penalty': 1.2, 'cut_words': 320}


## 3. Apply the chosen decoding to every experiment (150 local-eval + 100 test)


In [4]:
for label, cfg, submit in EXPERIMENTS:
    print(f"\n{'=' * 100}\n{time.strftime('%H:%M')}  {label}\n{'=' * 100}")
    exp = None
    try:
        exp = Experiment(ROOT, cfg)
        exp.evaluate_local(decode=BEST_DEC, tag=BEST_TAG)
        if submit:
            exp.predict_test(decode=BEST_DEC, tag=BEST_TAG)
    except Exception:
        print(f"{label} FAILED:")
        traceback.print_exc(file=sys.stdout)
    finally:
        del exp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


16:25  #1 A2 full FT
experiment: A2_full_doc900 | device cuda | precision torch.bfloat16
[cache] training already done -> loaded best epoch 2


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 292.97it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


local eval_nr4_rp1.2_cut320: {'n': 150, 'rouge1_f1': 0.3545, 'rouge2_f1': 0.157, 'rougeL_f1': 0.197, 'pred_words': 287.44, 'bertscore_p': 0.8541, 'bertscore_r': 0.8361, 'bertscore_f1': 0.8449, 'repeated_4gram_pct': 0.0, 'ref_repeated_4gram_pct': 3.7261, 'novel_2gram_pct': 32.2313, 'ref_novel_2gram_pct': 28.7816, 'copy_weight': 0.6052, 'stop_reasons': {'eos': 113, 'max_tokens': 4, 'max_words': 33}, 'sec_per_doc': 9.9829, 'gen_tokens_per_s': 41.9512, 'gen_peak_gb': 0.46}


test summaries_nr4_rp1.2_cut320: 100%|██████████| 100/100 [17:29<00:00, 10.49s/it]


submission check: {'docs': 100, 'mean_words': 288.5, 'min_words': 204, 'max_words': 320} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase6\A2_full_doc900\test_nr4_rp1.2_cut320\submission.zip

17:07  #2 B + LoRMA
experiment: B_lormar8_doc900 | device cuda | precision torch.bfloat16
[cache] training already done -> loaded best epoch 4


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 236.97it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (17 outputs left out)

test summaries_nr4_rp1.2_cut320: 100%|██████████| 100/100 [03:38<00:00,  2.19s/it]


submission check: {'docs': 100, 'mean_words': 289.2, 'min_words': 184, 'max_words': 320} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase6\Dsmall_dec024_full_enc1x1024_kd0.5\test_nr4_rp1.2_cut320\submission.zip

19:42  #6c D-small, no KD
experiment: Dsmall_dec024_full_enc1x1024 | device cuda | precision torch.bfloat16
[cache] training already done -> loaded best epoch 3


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 428.33it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


local eval_nr4_rp1.2_cut320: {'n': 150, 'rouge1_f1': 0.376, 'rouge2_f1': 0.168, 'rougeL_f1': 0.2037, 'pred_words': 300.74, 'bertscore_p': 0.8558, 'bertscore_r': 0.8372, 'bertscore_f1': 0.8463, 'repeated_4gram_pct': 0.0, 'ref_repeated_4gram_pct': 3.7261, 'novel_2gram_pct': 31.0231, 'ref_novel_2gram_pct': 28.7816, 'copy_weight': 0.5125, 'stop_reasons': {'eos': 45, 'max_words': 105}, 'sec_per_doc': 2.75, 'gen_tokens_per_s': 188.7855, 'gen_peak_gb': 0.31}


## 4. Before (P6_02 decoding) vs after (chosen decoding), local-eval 150 documents

Reference points (P6_01, same documents): lead-512 0.832, random sentences 0.842, oracle extractive 0.873.


In [5]:
rows = []
for label, cfg, submit in EXPERIMENTS:
    e = Experiment(ROOT, cfg, quiet=True)
    for name, f in (("P6_02 (nr8)", e.dir / "local_scores.json"), (BEST_TAG, e.dir / f"local_scores_{BEST_TAG}.json")):
        if f.exists():
            r = json.load(open(f))
            rows.append({"experiment": label, "decoding": name, **{k: r.get(k) for k in
                         ("bertscore_f1", "bertscore_p", "bertscore_r", "rouge1_f1", "rouge2_f1", "rougeL_f1", "pred_words",
                          "repeated_4gram_pct", "novel_2gram_pct", "copy_weight", "sec_per_doc")}})
final = pd.DataFrame(rows).set_index(["experiment", "decoding"])
display(final.round(4))
final.to_csv(ROOT / "experiments" / "phase6" / "final_local_eval.csv")
print("submission zips:")
for label, cfg, submit in EXPERIMENTS:
    z = Experiment(ROOT, cfg, quiet=True).dir / f"test_{BEST_TAG}" / "submission.zip"
    if submit:
        print(f"  {label:28s} {z.relative_to(ROOT)}  {'OK' if z.exists() else 'MISSING'}")

bertscore_f1  bertscore_p  bertscore_r  rouge1_f1  rouge2_f1  rougeL_f1  pred_words  repeated_4gram_pct  novel_2gram_pct  copy_weight  sec_per_doc
experiment               decoding                                                                                                                                                            
#1 A2 full FT            P6_02 (nr8)             0.8347       0.8422       0.8275     0.3491     0.1467     0.2018    351.7000             22.1808          30.4161       0.6907       4.5073
                         nr4_rp1.2_cut320        0.8449       0.8541       0.8361     0.3545     0.1570     0.1970    287.4400              0.0000          32.2313       0.6052       9.9829
#2 B + LoRMA             P6_02 (nr8)             0.8191       0.8261       0.8124     0.3225     0.1152     0.1949    418.6000             31.8507          49.0159          NaN       7.8518
                         nr4_rp1.2_cut320        0.8405       0.8494       0.8319     0.3499     0.1359     0.1851    285.0333              0.0000          48.1626          NaN      15.4054
#3 A + Houlsby           P6_02 (nr8)             0.8230       0.8314       0.8151     0.3235     0.1201     0.1948    409.7600             30.5431          47.9158          NaN       6.4653
                         nr4_rp1.2_cut320        0.8412       0.8501       0.8327     0.3487     0.1375     0.1843    281.6333              0.0000          48.2741          NaN       9.6269
#4 D + LoRA 1x1024       P6_02 (nr8)             0.8289       0.8371       0.8211     0.3506     0.1426     0.2050    412.4733             28.9732          35.3822       0.5214       5.7881
                         nr4_rp1.2_cut320        0.8421       0.8521       0.8325     0.3592     0.1556     0.1960    295.4200              0.0000          35.1575       0.4870       5.4273
#5 D + LoRA 3x1024 (FiD) P6_02 (nr8)             0.8397       0.8455       0.8340     0.3993     0.1802     0.2198    420.4600             22.3456          22.4167       0.6785       5.8768
                         nr4_rp1.2_cut320        0.8493       0.8574       0.8413     0.3832     0.1826     0.2103    293.6000              0.0000          19.9535       0.6884       5.6856
#6 D-small + KD          P6_02 (nr8)             0.8318       0.8400       0.8239     0.3467     0.1423     0.2002    384.3067             27.1951          34.4422       0.5221       2.2403
                         nr4_rp1.2_cut320        0.8452       0.8562       0.8345     0.3653     0.1621     0.2015    289.3800              0.0000          34.9559       0.4865       2.1733
#6c D-small, no KD       P6_02 (nr8)             0.8335       0.8389       0.8283     0.3744     0.1557     0.2102    448.4600             27.1664          32.0852       0.5640       2.6506
                         nr4_rp1.2_cut320        0.8463       0.8558       0.8372     0.3760     0.1680     0.2037    300.7400              0.0000          31.0231       0.5125       2.7500

submission zips:
  #1 A2 full FT                checkpoints\phase6\A2_full_doc900\test_nr4_rp1.2_cut320\submission.zip  OK
  #2 B + LoRMA                 checkpoints\phase6\B_lormar8_doc900\test_nr4_rp1.2_cut320\submission.zip  OK
  #3 A + Houlsby               checkpoints\phase6\A_houlsby64_doc900\test_nr4_rp1.2_cut320\submission.zip  OK
  #4 D + LoRA 1x1024           checkpoints\phase6\D_lorar8_enc1x1024\test_nr4_rp1.2_cut320\submission.zip  OK
  #5 D + LoRA 3x1024 (FiD)     checkpoints\phase6\D_lorar8_enc3x1024\test_nr4_rp1.2_cut320\submission.zip  OK
  #6 D-small + KD              checkpoints\phase6\Dsmall_dec024_full_enc1x1024_kd0.5\test_nr4_rp1.2_cut320\submission.zip  OK


## 5. Example: the same document before and after (#1 and #5, first 120 words)


In [6]:
for label, cfg, _ in SWEEP_MODELS:
    e = Experiment(ROOT, cfg, quiet=True)
    i = e.local_ids[0]
    for name, f in (("before (nr8)", e.dir / "local_preds.json"), (f"after ({BEST_TAG})", e.dir / f"local_preds_{BEST_TAG}.json")):
        if f.exists():
            print(f"\n{label} {name}:\n  ", " ".join(json.load(open(f))[i]["summary"].split()[:120]))
print("\nREFERENCE:\n  ", " ".join(su.clean_summ(e.by_id[i]["summary"]).split()[:120]))


#1 A2 full FT before (nr8):
   The appellant was the Manager of a tank which was a charitable institution. The Inam was granted to the appellant. The appellant was shown to be the Manager of the tank. The Inam was granted in favour of the tank and the appellant was in possession of the Inam property. The appellant was in possession of the Inam. The Inam was granted in favour by the Collector. The appellant was in possession of certain Inam property. The Inam was granted in favour the tank and the appellant was in the possession of the Inam property. The Inam was granted to the tank and the appellant was in occupation of the Inam property. The appellant filed a writ petition in the

#1 A2 full FT after (nr4_rp1.2_cut320):
   The appellant was the Manager of a tank which was a charitable institution. The Inam was granted to the appellant by the Collector of Andhra Pradesh. The appellant was shown to be the Manager of the tank and the Inam was shown to have been granted to the tank. The 